In [ ]:
# --- CELDA 1: Importaciones y Configuracion ---
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Configuracion
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_PATH = './dataset'

In [ ]:
# --- CELDA 2: Preprocesamiento (Data Augmentation) ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False # Importante para la matriz
)

In [ ]:
# --- CELDA 3: Crear el Modelo y Compilar (Versión Mejorada) ---
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# 1. Cargar Base (MobileNetV2)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Configuracion de Fine-Tuning
base_model.trainable = True
# Congelamos las primeras 100 capas (MobileNet tiene 155 en total)
for layer in base_model.layers[:100]:
    layer.trainable = False

# 3. Construir el "Cerebro"
x = base_model.output
x = GlobalAveragePooling2D()(x)

# Capa densa grande para aprender formas complejas
x = Dense(512, activation='relu')(x) 
x = Dropout(0.5)(x)  # Apaga el 50% de neuronas para evitar memorizacion

# Capa de refinamiento
x = Dense(256, activation='relu')(x) 

# Capa de salida (Clasificacion final)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

# 4. CREAR EL MODELO
model = Model(inputs=base_model.input, outputs=predictions)

# 5. COMPILAR (Aqui va la tasa de aprendizaje baja)
# Usamos 1e-5 (0.00001) para que el ajuste sea suave y preciso
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("Modelo creado y compilado para Fine-Tuning.")

In [ ]:
# --- CELDA 4: Entrenar ---
print("Iniciando entrenamiento...")
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=validation_generator
)

In [ ]:
# --- CELDA 5: Guardar el Modelo ---
model.save('./modelo/modelo_residuos.h5')
print("Modelo guardado como 'modelo_residuos.h5'")

In [ ]:
# --- CELDA 6: Evaluacion y Graficas ---
# Grafica de precision

plt.plot(history.history['accuracy'], label='Precision con ejercicios de practica')
plt.plot(history.history['val_accuracy'], label = 'Precision con ejercicios reales')
plt.xlabel('Epoca')
plt.ylabel('Precision')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()

# Matriz de confusion
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)
cm = confusion_matrix(validation_generator.classes, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=validation_generator.class_indices.keys(),
            yticklabels=validation_generator.class_indices.keys())
plt.ylabel('Verdadero')
plt.xlabel('Prediccion')
plt.show()